# Uncertainty bands: Module B vs Module C (pIRF) — RF & AGWP (2-column grid)

- **Purpose**: Compare *uncertainty bands* (5–95%) and medians across ensemble members between:
  - **Module B** (analytical)
  - **Module C** (pIRF-based; optionally `apply_cc=True` depending on the Excel exported)

- **Layout**: one row per **scenario**, two columns:
  1. **RF**
  2. **AGWP**

- **Dynamic rows**: number of scenarios is driven by the `SCENARIOS` list.
- **Model year**: controlled by `MODEL_YEAR`.

Generated on 2026-03-04.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Inputs

In [ ]:
# ---------------------------
# User inputs
# ---------------------------

MODEL_YEAR = 2050
H_MAX_PLOT = 100  # plot horizon 0..H_MAX_PLOT

# Add/remove scenarios here (rows will adapt automatically)
SCENARIOS = ["ssp585"]  # e.g., ["ssp119", "ssp126", "ssp245", "ssp585"]

# Map each scenario to the two Excel files to compare.
# Option A: provide explicit file paths (recommended).
# Option B: build paths from a folder + naming patterns.
#
# Required: each Excel must contain sheets like:
#   - rf_wh_ensmb_CO2{MODEL_YEAR}
#   - agwp_wh_ensmb_CO2{MODEL_YEAR}
#
# Example (edit paths):
FILES_BY_SCN = {
    "ssp585": {
        "moduleB": "/path/to/moduleB_CO2_ssp585_MY2050.xlsx",
        "moduleC": "/path/to/moduleC_CO2_ssp585_MY2050_applyccTrue.xlsx",
    },
    # "ssp245": {"moduleB": "...", "moduleC": "..."},
}

# Labels used in legends
LABEL_B = "Module B"
LABEL_C = "Module C"


## Helpers

In [ ]:
def _find_sheet(xls: pd.ExcelFile, key_substr: str) -> str:
    hits = [s for s in xls.sheet_names if key_substr.lower() in s.lower()]
    if not hits:
        raise KeyError(f"Sheet containing '{key_substr}' not found. Available sheets: {xls.sheet_names}")
    return hits[0]

def load_ensemble_sheet(xlsx_path: str, sheet_key_substr: str):
    """Load an ensemble sheet with first col = H and remaining cols = ensemble members."""
    xls = pd.ExcelFile(xlsx_path)
    sheet = _find_sheet(xls, sheet_key_substr)
    df = pd.read_excel(xlsx_path, sheet_name=sheet)
    H = df.iloc[:, 0].to_numpy(dtype=float)
    X = df.iloc[:, 1:].to_numpy(dtype=float)  # shape (H, ens)
    return H, X, sheet

def align_on_common_H(H1, X1, H2, X2, H_max_plot=100):
    """Align two (H, X) pairs onto common H values (<= H_max_plot)."""
    common = np.intersect1d(H1, H2)
    common = common[common <= H_max_plot]
    if common.size == 0:
        raise ValueError("No common H found between the two files within plot range.")
    idx1 = {h:i for i,h in enumerate(H1)}
    idx2 = {h:i for i,h in enumerate(H2)}
    rows1 = [idx1[h] for h in common]
    rows2 = [idx2[h] for h in common]
    return common, X1[rows1, :], X2[rows2, :]

def summarize_band(X, qlo=5, qhi=95):
    """Return median, qlo, qhi over ensemble axis (axis=1)."""
    med = np.nanmedian(X, axis=1)
    lo  = np.nanpercentile(X, qlo, axis=1)
    hi  = np.nanpercentile(X, qhi, axis=1)
    return med, lo, hi


## Plot grid (2 columns: RF | AGWP)

In [ ]:
# ---------------------------
# Main plotting routine
# ---------------------------

def plot_grid_co2_rf_agwp(
    scenarios,
    files_by_scn,
    model_year=2050,
    h_max_plot=100,
    label_b="Module B",
    label_c="Module C",
    qlo=5,
    qhi=95,
):
    n = len(scenarios)
    if n == 0:
        raise ValueError("No scenarios provided.")

    fig, axes = plt.subplots(nrows=n, ncols=2, figsize=(12, 3.5 * n), sharex=False)
    if n == 1:
        # make axes indexable as [row, col]
        axes = np.array([axes])

    for i, scn in enumerate(scenarios):
        if scn not in files_by_scn:
            raise KeyError(f"Scenario '{scn}' not found in FILES_BY_SCN.")

        fB = files_by_scn[scn]["moduleB"]
        fC = files_by_scn[scn]["moduleC"]

        # --- RF ---
        H_B, X_B, shB = load_ensemble_sheet(fB, f"rf_wh_ensmb_CO2{model_year}")
        H_C, X_C, shC = load_ensemble_sheet(fC, f"rf_wh_ensmb_CO2{model_year}")
        H, Xb, Xc = align_on_common_H(H_B, X_B, H_C, X_C, H_max_plot=h_max_plot)
        b_med, b_lo, b_hi = summarize_band(Xb, qlo=qlo, qhi=qhi)
        c_med, c_lo, c_hi = summarize_band(Xc, qlo=qlo, qhi=qhi)

        ax = axes[i, 0]
        ax.plot(H, b_med, label=f"{label_b} (median)")
        ax.fill_between(H, b_lo, b_hi, alpha=0.2, label=f"{label_b} ({qlo}–{qhi}%)")
        ax.plot(H, c_med, label=f"{label_c} (median)")
        ax.fill_between(H, c_lo, c_hi, alpha=0.2, label=f"{label_c} ({qlo}–{qhi}%)")
        ax.set_title(f"CO$_2$ {scn.upper()} MY{model_year} — RF")
        ax.set_xlabel("Horizon (years)")
        ax.set_ylabel("RF (W m$^{-2}$)")
        ax.grid(True, alpha=0.3)

        # --- AGWP ---
        H_B, X_B, shB = load_ensemble_sheet(fB, f"agwp_wh_ensmb_CO2{model_year}")
        H_C, X_C, shC = load_ensemble_sheet(fC, f"agwp_wh_ensmb_CO2{model_year}")
        H, Xb, Xc = align_on_common_H(H_B, X_B, H_C, X_C, H_max_plot=h_max_plot)
        b_med, b_lo, b_hi = summarize_band(Xb, qlo=qlo, qhi=qhi)
        c_med, c_lo, c_hi = summarize_band(Xc, qlo=qlo, qhi=qhi)

        ax = axes[i, 1]
        ax.plot(H, b_med, label=f"{label_b} (median)")
        ax.fill_between(H, b_lo, b_hi, alpha=0.2, label=f"{label_b} ({qlo}–{qhi}%)")
        ax.plot(H, c_med, label=f"{label_c} (median)")
        ax.fill_between(H, c_lo, c_hi, alpha=0.2, label=f"{label_c} ({qlo}–{qhi}%)")
        ax.set_title(f"CO$_2$ {scn.upper()} MY{model_year} — AGWP")
        ax.set_xlabel("Horizon (years)")
        ax.set_ylabel("AGWP (W m$^{-2}$ yr)")
        ax.grid(True, alpha=0.3)

        # Put legends only on the first row to reduce clutter
        if i == 0:
            axes[i, 0].legend(loc="best", frameon=True)
            axes[i, 1].legend(loc="best", frameon=True)

    fig.tight_layout()
    return fig


# ---- Run ----
fig = plot_grid_co2_rf_agwp(
    scenarios=SCENARIOS,
    files_by_scn=FILES_BY_SCN,
    model_year=MODEL_YEAR,
    h_max_plot=H_MAX_PLOT,
    label_b=LABEL_B,
    label_c=LABEL_C,
    qlo=5,
    qhi=95,
)
plt.show()


## Notes / Next steps
- Add more scenarios by extending `SCENARIOS` and `FILES_BY_SCN`.
- Change `MODEL_YEAR` to 2030/2040/etc. The sheet keys will update automatically.
- To include additional metrics (AGTP/iAGTP/GWP), replicate the two-column pattern or expand columns.
